# WatchDog: train the weapon detector

Trains a YOLOv8 model to flag two things: **gun** and **knife**. Training data is Google's Open Images (annotations CC BY 4.0, images listed as CC BY 2.0), plus look-alike photos with no weapon (umbrellas, phones, tools, drills, hair dryers, instruments, cases, ladders) so the model learns what *isn't* a weapon.

**What changed in this version, and why.** The hard-case test (see the README) showed two kinds of mistake: false alarms on long, dark, hand-held things (drills, tripods, a saxophone case, a cart), and misses on small, upright guns in CCTV-style footage. So this notebook now trains the slightly bigger **YOLOv8s** (instead of the smallest, YOLOv8n) for longer, with more size variation (`scale=0.7`) and some rotation (`degrees=10`) so small and tilted guns are seen in training, and half as many look-alike photos as weapon photos (`--lookalike-ratio 0.5`, was 0.3). It also exports the ONNX file the web demo runs in the browser.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.

It takes roughly 1.5 to 2 hours. At the end your browser downloads `weapon_model.zip`. Unzip it into `WatchDog/tracking/` on your laptop, and copy `weapons.onnx` from it to `WatchDog/public/models/weapons.onnx` for the web demo.


In [ ]:
!nvidia-smi -L  # should list a GPU; if it errors, set the runtime to T4 GPU (see above)

In [ ]:
!pip install -q ultralytics openvino

## 1. Build the dataset (~10-15 min)
This cell saves the dataset script; the next one runs it.

In [ ]:
%%writefile prepare_weapon_data.py
"""Build a YOLO weapon-detection dataset from Google's Open Images.

Labels:
    0 gun    (Open Images: Handgun, Rifle, Shotgun)
    1 knife  (Open Images: Knife, Kitchen knife, Dagger)

Also adds look-alike photos with no weapon in them (umbrellas, phones, tools,
drills, hair dryers, instruments, cases, school supplies...) as background images, so the model learns what *isn't* a weapon.

Open Images annotations are CC BY 4.0; images are listed as CC BY 2.0.
https://storage.googleapis.com/openimages/web/index.html

Usage (made for Google Colab; the training annotations file is ~2.3 GB):
    python prepare_weapon_data.py --out /content/weapons
    python prepare_weapon_data.py --out /tmp/weapons --quick   # small smoke test
"""

import argparse
import random
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from pathlib import Path

import pandas as pd
from PIL import Image

ANNOTATIONS = {
    "train": "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv",
    "validation": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
    "test": "https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv",
}
IMAGE_URL = "https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg"

CLASS_NAMES = ["gun", "knife"]
WEAPON_LABELS = {
    "/m/0gxl3": 0,  # Handgun
    "/m/06c54": 0,  # Rifle
    "/m/06nrc": 0,  # Shotgun
    "/m/04ctx": 1,  # Knife
    "/m/058qzx": 1,  # Kitchen knife
    "/m/02gzp": 1,  # Dagger
}
# Images with these boxes are skipped: they're weapons we don't label, and
# leaving them unlabeled would teach the model they're background.
SKIP_LABELS = {
    "/m/083kb",  # Weapon (unspecified)
    "/m/06y5r",  # Sword
}
LOOKALIKE_LABELS = {
    "/m/0hnnb",  # Umbrella
    "/m/050k8",  # Mobile phone
    "/m/01kb5b",  # Flashlight
    "/m/03l9g",  # Hammer
    "/m/01j5ks",  # Wrench
    "/m/01bms0",  # Screwdriver
    "/m/0qjjc",  # Remote control
    "/m/073bxn",  # Tripod
    "/m/0dv5r",  # Camera
    # Added after the hard-case test (tracking/eval): these were flagged as guns.
    "/m/01d380",  # Drill (Tool)
    "/m/03wvsk",  # Hair dryer
    "/m/06ncr",  # Saxophone
    "/m/0342h",  # Guitar
    "/m/07y_7",  # Violin
    "/m/01xqw",  # Cello
    "/m/07gql",  # Trumpet
    "/m/07c6l",  # Trombone
    "/m/0l14j_",  # Flute
    "/m/0584n8",  # Briefcase
    "/m/01s55n",  # Suitcase
    "/m/0lt4_",  # Binoculars
    "/m/0hg7b",  # Microphone
    # Things students carry on campus.
    "/m/01940j",  # Backpack
    "/m/01c648",  # Laptop
    "/m/0bh9flk",  # Tablet computer
    "/m/0bt_c3",  # Book
    "/m/0k1tl",  # Pen
    "/m/05676x",  # Pencil case
    "/m/0hdln",  # Ruler
    "/m/01lsmm",  # Scissors
    "/m/025fsf",  # Stapler
    "/m/024d2",  # Calculator
    "/m/04dr76w",  # Bottle
    "/m/01b7fy",  # Headphones
    "/m/02p5f1q",  # Coffee cup
    "/m/080hkjn",  # Handbag
    "/m/03g8mr",  # Baseball bat
    "/m/0h8my_4",  # Tennis racket
    "/m/06_fw",  # Skateboard
    "/m/05441v",  # Crutch
    # Added after the browser demo flagged a hospital cart and upright poles.
    "/m/012w5l",  # Ladder
}

DUPLICATE_IOU = 0.8  # same-class boxes overlapping this much are one object
MAX_SIDE = 640  # images are shrunk to this on download; YOLO trains at 640
SEED = 0


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--out", required=True, help="output folder for the dataset")
    parser.add_argument("--cache", default="oi_cache", help="where to keep annotation CSVs")
    parser.add_argument(
        "--lookalike-ratio",
        type=float,
        default=0.5,
        help="look-alike images to add, as a fraction of weapon images (default 0.5)",
    )
    parser.add_argument("--workers", type=int, default=32, help="parallel image downloads")
    parser.add_argument(
        "--quick", action="store_true", help="tiny dataset from the validation split only"
    )
    return parser.parse_args()


def download(url, path):
    if not path.exists():
        print(f"Downloading {url} ...")
        tmp = path.with_suffix(".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    return path


def load_boxes(csv_path):
    """Return the rows for labels we care about, read in chunks to save memory."""
    wanted = set(WEAPON_LABELS) | SKIP_LABELS | LOOKALIKE_LABELS
    columns = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax"]
    chunks = [
        chunk[chunk["LabelName"].isin(wanted)]
        for chunk in pd.read_csv(csv_path, usecols=columns, chunksize=2_000_000)
    ]
    return pd.concat(chunks)


def iou(a, b):
    ix = max(min(a.XMax, b.XMax) - max(a.XMin, b.XMin), 0)
    iy = max(min(a.YMax, b.YMax) - max(a.YMin, b.YMin), 0)
    inter = ix * iy
    union = (a.XMax - a.XMin) * (a.YMax - a.YMin) + (b.XMax - b.XMin) * (b.YMax - b.YMin) - inter
    return inter / union if union > 0 else 0


def weapon_boxes(rows):
    """Weapon rows with duplicates removed.

    One object is sometimes tagged twice (e.g. both Rifle and Shotgun), which
    becomes two "gun" boxes once the labels are merged.
    """
    kept = []
    for row in rows:
        if row.LabelName not in WEAPON_LABELS:
            continue
        cls = WEAPON_LABELS[row.LabelName]
        if not any(
            WEAPON_LABELS[k.LabelName] == cls and iou(row, k) > DUPLICATE_IOU for k in kept
        ):
            kept.append(row)
    return kept


def pick_images(boxes, lookalike_ratio, rng):
    """Return {image_id: [yolo label lines]}; look-alike images get no lines."""
    by_image = defaultdict(list)
    for row in boxes.itertuples(index=False):
        by_image[row.ImageID].append(row)

    weapons, lookalikes = {}, defaultdict(list)  # look-alike label -> image ids
    for image_id, rows in by_image.items():
        labels = {row.LabelName for row in rows}
        if labels & SKIP_LABELS:
            continue
        if labels & set(WEAPON_LABELS):
            weapons[image_id] = [
                "{} {:.6f} {:.6f} {:.6f} {:.6f}".format(
                    WEAPON_LABELS[row.LabelName],
                    (row.XMin + row.XMax) / 2,
                    (row.YMin + row.YMax) / 2,
                    row.XMax - row.XMin,
                    row.YMax - row.YMin,
                )
                for row in weapon_boxes(rows)
            ]
        else:
            for label in labels:
                lookalikes[label].append(image_id)

    # Take one image of each kind in turn, so rare look-alikes (drills, hair
    # dryers) aren't crowded out by thousands of books and bottles.
    for ids in lookalikes.values():
        rng.shuffle(ids)
    count = int(len(weapons) * lookalike_ratio)
    chosen = {}
    queues = [lookalikes[label] for label in sorted(lookalikes)]
    while len(chosen) < count and any(queues):
        for ids in queues:
            while ids and ids[-1] in chosen:
                ids.pop()
            if ids and len(chosen) < count:
                chosen[ids.pop()] = []
    return weapons | chosen


def save_image(split, image_id, path):
    if path.exists():
        return True
    try:
        url = IMAGE_URL.format(split=split, image_id=image_id)
        with urllib.request.urlopen(url, timeout=30) as response:
            image = Image.open(BytesIO(response.read())).convert("RGB")
        image.thumbnail((MAX_SIDE, MAX_SIDE))
        image.save(path, quality=90)
        return True
    except Exception as error:
        print(f"  skipped {image_id}: {error}")
        return False


def build_split(name, sources, out, args, rng):
    """sources: [(open_images_split, csv_path), ...] merged into one YOLO split."""
    images_dir = out / "images" / name
    labels_dir = out / "labels" / name
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    jobs = []
    for oi_split, csv_path in sources:
        picked = pick_images(load_boxes(csv_path), args.lookalike_ratio, rng)
        if args.quick:
            picked = dict(list(picked.items())[:20])
        jobs += [(oi_split, image_id, lines) for image_id, lines in picked.items()]

    print(f"{name}: downloading {len(jobs)} images ...")
    with ThreadPoolExecutor(args.workers) as pool:
        results = pool.map(
            lambda job: save_image(job[0], job[1], images_dir / f"{job[1]}.jpg"), jobs
        )
        saved = 0
        for (oi_split, image_id, lines), ok in zip(jobs, results):
            if ok:
                (labels_dir / f"{image_id}.txt").write_text("\n".join(lines))
                saved += 1

    counts = defaultdict(int)
    for _, _, lines in jobs:
        for line in lines:
            counts[CLASS_NAMES[int(line.split()[0])]] += 1
        if not lines:
            counts["look-alike images"] += 1
    print(f"{name}: saved {saved} images; " + ", ".join(f"{k} {v}" for k, v in counts.items()))


def main():
    args = parse_args()
    out = Path(args.out)
    cache = Path(args.cache)
    cache.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEED)

    def csv(split):
        return download(ANNOTATIONS[split], cache / f"{split}.csv")

    if args.quick:
        build_split("train", [("validation", csv("validation"))], out, args, rng)
        build_split("val", [("test", csv("test"))], out, args, rng)
    else:
        build_split("train", [("train", csv("train"))], out, args, rng)
        build_split("val", [("validation", csv("validation")), ("test", csv("test"))], out, args, rng)

    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n" + "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASS_NAMES))
    )
    print(f"Dataset ready: {out / 'data.yaml'}")


if __name__ == "__main__":
    main()


In [ ]:
!python prepare_weapon_data.py --out /content/weapons --cache /content/oi_cache --lookalike-ratio 0.5


## 1b. Optional: add your own CCTV frames or extra datasets

The Open Images photos are mostly close-ups. To make the model good at hallway-camera footage, add frames that look like it. Put any of these in a folder called **`WatchDog/extra_data`** in your Google Drive, then run the three cells below:

- the output folder of `extract_frames.py` (your own clips, boxes fixed in a labeling tool), zipped or not
- a Roboflow "YOLOv8" export `.zip`
- any folder with `images/` and `labels/` in YOLO format

Extra images are repeated 5 times in the training list so a few hundred of them count. If the folder is empty or missing, nothing changes.

To **fine-tune a model you already trained** instead of starting from scratch (faster, keeps what it knows), also put its `weapons.pt` in that Drive folder: the training cell then starts from it and runs 30 epochs instead of 80. See `tracking/train/README.md` for the full walkthrough.


In [ ]:
%%writefile merge_datasets.py
"""Fold extra YOLO datasets (your own CCTV frames, Roboflow exports) into the training dataset.

Takes the Open Images dataset that prepare_weapon_data.py builds (or an empty folder) and
adds one or more extra datasets to it, in the same layout the training notebook reads:

    DATASET/images/train, DATASET/images/val, DATASET/labels/train, DATASET/labels/val,
    DATASET/data.yaml

Each extra dataset can be:
  - the output folder of extract_frames.py (images/, labels/, classes.txt)
  - a Roboflow "YOLOv8" export, as a .zip or an unzipped folder (train/, valid/, test/, data.yaml)
  - any folder with images/ + labels/ (+ optional train/val subfolders, data.yaml or classes.txt)

Class names are read from each dataset's data.yaml or classes.txt and mapped onto WatchDog's
classes (gun, knife) with --map; classes not in the map are dropped from the labels. Datasets
without a train/val split get an 85/15 split. --oversample repeats the extra images in the
train list so a few hundred CCTV frames are not drowned out by thousands of stock photos.

Usage:
    python merge_datasets.py --dataset /content/weapons --extra my_frames --oversample 5
    python merge_datasets.py --dataset /content/weapons --extra guns.v3.zip --extra my_frames \\
        --map "pistol=gun,rifle=gun,handgun=gun,weapon=gun,knife=knife" --oversample 5
"""

import argparse
import random
import re
import shutil
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

CLASS_NAMES = ["gun", "knife"]
DEFAULT_MAP = (
    "gun=gun,guns=gun,pistol=gun,pistols=gun,handgun=gun,handguns=gun,rifle=gun,rifles=gun,"
    "shotgun=gun,firearm=gun,firearms=gun,weapon=gun,weapons=gun,"
    "knife=knife,knives=knife,dagger=knife,blade=knife"
)
IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VAL_SHARE = 0.15
SEED = 0


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--dataset", required=True, help="dataset folder to add to (created if missing)")
    parser.add_argument("--extra", action="append", required=True, help="extra dataset (folder or .zip); repeatable")
    parser.add_argument("--map", default=DEFAULT_MAP, help='class name mapping, e.g. "pistol=gun,knife=knife"')
    parser.add_argument("--oversample", type=int, default=1, help="repeat extra train images this many times")
    parser.add_argument("--val-share", type=float, default=VAL_SHARE, help="val share for unsplit datasets")
    return parser.parse_args()


def parse_map(text):
    mapping = {}
    for pair in text.split(","):
        if "=" not in pair:
            continue
        src, dst = (s.strip().lower() for s in pair.split("=", 1))
        if dst not in CLASS_NAMES:
            sys.exit(f"--map: '{dst}' is not one of {CLASS_NAMES}")
        mapping[src] = CLASS_NAMES.index(dst)
    return mapping


def read_names(root):
    """Class names of a dataset, from data.yaml (names: list or dict) or classes.txt."""
    for yaml_path in list(root.glob("*.yaml")) + list(root.glob("*.yml")):
        text = yaml_path.read_text(errors="ignore")
        m = re.search(r"^names:\s*(\[.*\])\s*$", text, re.M)
        if m:
            return [n.strip().strip("'\"") for n in m.group(1).strip("[]").split(",") if n.strip()]
        m = re.search(r"^names:\s*\n((?:\s+-?\s*\S.*\n?)+)", text, re.M)
        if m:
            names = {}
            for line in m.group(1).splitlines():
                item = re.match(r"\s+(?:(\d+)\s*:\s*|-\s*)(.+)$", line)
                if item:
                    idx = int(item.group(1)) if item.group(1) else len(names)
                    names[idx] = item.group(2).strip().strip("'\"")
            return [names[i] for i in sorted(names)]
    classes = root / "classes.txt"
    if classes.exists():
        return [l.strip() for l in classes.read_text().splitlines() if l.strip()]
    return None


def find_splits(root):
    """{split: [(image, label_or_None)]} for whatever layout the dataset uses."""
    splits = defaultdict(list)

    def pair_up(images_dir, labels_dir, split):
        for img in sorted(images_dir.rglob("*")):
            if img.suffix.lower() not in IMAGE_TYPES:
                continue
            rel = img.relative_to(images_dir).with_suffix(".txt")
            label = labels_dir / rel
            splits[split].append((img, label if label.exists() else None))

    for name, split in (("train", "train"), ("valid", "val"), ("val", "val"), ("test", "val")):
        sub = root / name
        if (sub / "images").is_dir():
            pair_up(sub / "images", sub / "labels", split)
    if splits:
        return splits
    images, labels = root / "images", root / "labels"
    if (images / "train").is_dir():
        pair_up(images / "train", labels / "train", "train")
        for v in ("val", "valid", "test"):
            if (images / v).is_dir():
                pair_up(images / v, labels / v, "val")
        return splits
    if images.is_dir():
        pair_up(images, labels, "unsplit")
    return splits


def convert_label(label, names, mapping, counts):
    """Rewrite a YOLO label file onto WatchDog's class ids; returns the new lines."""
    lines = []
    if label is None:
        return lines
    for line in label.read_text(errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            idx = int(float(parts[0]))
        except ValueError:
            continue
        name = names[idx].lower() if names and idx < len(names) else CLASS_NAMES[idx] if idx < len(CLASS_NAMES) else None
        if name is None or name not in mapping:
            counts["dropped:" + str(name)] += 1
            continue
        cls = mapping[name]
        lines.append(" ".join([str(cls)] + parts[1:5]))
        counts[CLASS_NAMES[cls]] += 1
    return lines


def unpack(extra, tmp):
    path = Path(extra)
    if path.suffix.lower() == ".zip":
        target = tmp / path.stem
        with zipfile.ZipFile(path) as z:
            z.extractall(target)
        inner = [p for p in target.iterdir() if p.is_dir()]
        return inner[0] if len(inner) == 1 and not (target / "data.yaml").exists() else target
    return path


def main():
    args = parse_args()
    mapping = parse_map(args.map)
    rng = random.Random(SEED)
    dataset = Path(args.dataset)
    for split in ("train", "val"):
        (dataset / "images" / split).mkdir(parents=True, exist_ok=True)
        (dataset / "labels" / split).mkdir(parents=True, exist_ok=True)
    existing = {p.name for split in ("train", "val") for p in (dataset / "images" / split).iterdir()}
    oversampled = []
    summary = {}

    with tempfile.TemporaryDirectory() as tmp:
        for extra in args.extra:
            root = unpack(extra, Path(tmp))
            source = Path(extra).name
            names = read_names(root)
            splits = find_splits(root)
            if not splits:
                print(f"{source}: no images found (expected images/ or train/images/), skipping")
                continue
            if "unsplit" in splits:
                items = splits.pop("unsplit")
                rng.shuffle(items)
                n_val = int(len(items) * args.val_share)
                splits["val"] = items[:n_val]
                splits["train"] = items[n_val:]
            counts = Counter()
            added = Counter()
            prefix = re.sub(r"[^A-Za-z0-9]+", "_", Path(extra).stem)[:40]
            for split, items in splits.items():
                for img, label in items:
                    name = f"{prefix}__{img.stem}"
                    if f"{name}{img.suffix.lower()}" in existing:
                        counts["duplicates"] += 1
                        continue
                    existing.add(f"{name}{img.suffix.lower()}")
                    lines = convert_label(label, names, mapping, counts)
                    dest = dataset / "images" / split / f"{name}{img.suffix.lower()}"
                    shutil.copy(img, dest)
                    (dataset / "labels" / split / f"{name}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
                    added[split] += 1
                    if not lines:
                        counts["background images"] += 1
                    if split == "train":
                        oversampled.append(dest)
            summary[source] = (added, counts, names)

    # Oversampling: Ultralytics reads train images from a .txt list if data.yaml points at one.
    train_list = dataset / "train.txt"
    base_images = sorted(
        p for p in (dataset / "images" / "train").iterdir()
        if p.suffix.lower() in IMAGE_TYPES and p not in set(oversampled)
    )
    with train_list.open("w") as f:
        for p in base_images:
            f.write(f"{p.resolve()}\n")
        for _ in range(max(1, args.oversample)):
            for p in oversampled:
                f.write(f"{p.resolve()}\n")
    (dataset / "data.yaml").write_text(
        f"path: {dataset.resolve()}\n"
        "train: train.txt\n"
        "val: images/val\n"
        "names:\n" + "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASS_NAMES))
    )

    print()
    for source, (added, counts, names) in summary.items():
        print(f"{source}: classes found {names or '(none; assumed gun,knife)'}")
        print(f"  added {added['train']} train + {added['val']} val images; "
              + ", ".join(f"{k} {v}" for k, v in sorted(counts.items())))
    print(f"\nTrain list: {len(base_images)} existing + {len(oversampled)} extra x{max(1, args.oversample)} "
          f"= {len(base_images) + len(oversampled) * max(1, args.oversample)} lines in {train_list}")
    print(f"Dataset ready: {dataset / 'data.yaml'}")


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import subprocess

EXTRA_DIR = Path("/content/drive/MyDrive/WatchDog/extra_data")
OVERSAMPLE = 5
RESUME_FROM = None  # set below if extra_data/weapons.pt exists

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as error:  # not on Colab, or Drive declined
    print("Google Drive not mounted:", error)

extras = sorted(p for p in EXTRA_DIR.glob("*") if p.is_dir() or p.suffix.lower() == ".zip") if EXTRA_DIR.exists() else []
if extras:
    print("Adding:", ", ".join(p.name for p in extras))
    command = ["python", "merge_datasets.py", "--dataset", "/content/weapons", "--oversample", str(OVERSAMPLE)]
    for extra in extras:
        command += ["--extra", str(extra)]
    subprocess.run(command, check=True)
else:
    print(f"No extra datasets in {EXTRA_DIR}; training on Open Images only.")

if (EXTRA_DIR / "weapons.pt").exists():
    RESUME_FROM = str(EXTRA_DIR / "weapons.pt")
    print("Will fine-tune from", RESUME_FROM)


## 2. Train (~1-1.5 h; ~30 min when fine-tuning)
Stops early if it stops improving for 15 epochs.


In [ ]:
from ultralytics import YOLO

# Start from the standard model and teach it weapons, or fine-tune your own (see 1b).
resume_from = globals().get("RESUME_FROM")
model = YOLO(resume_from or "yolov8s.pt")
model.train(
    data="/content/weapons/data.yaml",
    epochs=30 if resume_from else 80,  # fine-tuning needs fewer
    patience=15,
    imgsz=640,
    batch=16,  # yolov8s is about three times the size of yolov8n; 16 fits a T4
    hsv_v=0.6,  # brightness varies +/-60% (default 40%) so dark scenes look familiar
    scale=0.7,  # images are shrunk/grown by up to 70% (default 50%) so small, distant guns are seen
    degrees=10,  # rotate up to +/-10 degrees: guns are not always level
    flipud=0.0,  # no upside-down images; upright guns are the hard case, not inverted ones
    project="/content/runs",
    name="weapons",
    exist_ok=True,
)


## 3. Check how well it does
`P` (precision) is how often a flag is really a weapon; `R` (recall) is how many weapons it catches. Per-class lines show gun and knife separately.

In [ ]:
best = YOLO("/content/runs/weapons/weights/best.pt")
metrics = best.val(data="/content/weapons/data.yaml", imgsz=640)

## 4. Convert for the laptop and the browser, then download
The web demo (`src/detector.js`) runs `weapons.onnx` with ONNX Runtime Web and decodes the raw `(1, 6, 8400)` output itself, so the ONNX export is fixed at 640x640, batch 1, with no built-in NMS. After unzipping, copy `weapons.onnx` to `WatchDog/public/models/weapons.onnx`.


In [ ]:
import shutil
from pathlib import Path

out = Path("/content/weapon_model")
shutil.rmtree(out, ignore_errors=True)
out.mkdir()

# OpenVINO runs fast on Intel CPUs; dynamic=True lets the laptop pick the input size.
exported = Path(best.export(format="openvino", dynamic=True))
shutil.copytree(exported, out / "weapons_openvino_model")

# ONNX for the browser: fixed 640x640 input, batch 1, raw output (the demo does its own NMS).
onnx_path = Path(best.export(format="onnx", imgsz=640, batch=1, dynamic=False, simplify=True))
shutil.copy(onnx_path, out / "weapons.onnx")

shutil.copy("/content/runs/weapons/weights/best.pt", out / "weapons.pt")
shutil.copy("/content/runs/weapons/results.png", out / "training_results.png")
shutil.make_archive("/content/weapon_model", "zip", out)

from google.colab import files
files.download("/content/weapon_model.zip")
